# Institutional Futures Data State Research

## Lift 1 scope

This notebook inspects point-in-time data availability, continuous/mapped/actual-contract identity, contract coverage, mapping observations, and semantic sessions for ES, ZN, and 6E. Explicitly, **no strategy or P&L is being tested.** It produces no signal, return, indicator, forecast, position, order, or trading conclusion. Backwards Ratio adjusted values are used only for identity and coverage because official QC documentation says the adjustment uses full futures history.

In [ ]:
# Environment and version reporting
from __future__ import annotations

import platform
import sys

environment_report = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "lift": 1,
    "qc_runtime_execution_required": True,
}
environment_report

In [ ]:
# Market registry
from systematic_futures.config.markets import reference_market_definitions

markets = reference_market_definitions()
[(market.root, market.qc_future_constant_path, market.exchange_timezone) for market in markets]

In [ ]:
# Build a QuantBook
# QuantBook is provided by the verified QuantConnect Research runtime.
qb = QuantBook()
qb.set_time_zone("UTC")
# Cap interactive history access after the fixed 2024-03-25 probe end.
qb.set_start_date(2024, 3, 26)
qb

In [ ]:
# Register ES, ZN, and 6E
from systematic_futures.research_lib.quantbook_probe import add_reference_futures

subscriptions = add_reference_futures(qb, markets)
tuple(subscriptions)

In [ ]:
# Request fixed-period history
from datetime import UTC, datetime

from systematic_futures.research_lib.quantbook_probe import request_reference_history

start_utc = datetime(2024, 2, 15, 0, 0, tzinfo=UTC)
end_utc = datetime(2024, 3, 25, 23, 59, tzinfo=UTC)
histories = request_reference_history(qb, subscriptions, start_utc, end_utc)
tuple(histories)

In [ ]:
# Coverage inspection
from systematic_futures.research_lib.coverage_report import summarize_history_coverage

coverage = {
    root: summarize_history_coverage(root, histories[root])
    for root in subscriptions
}
coverage

In [ ]:
# Continuous and actual contract identity
identity_comparison = {
    root: {
        "continuous_symbol": coverage[root]["continuous_symbol"],
        "mapped_contracts_observed": coverage[root]["mapped_contracts_observed"],
        "expiry_coverage": coverage[root]["expiry_coverage"],
    }
    for root in subscriptions
}
identity_comparison

In [ ]:
# Mapping events and roll-window observations
mapping_and_roll_observations = {
    root: {
        "mapping_event_count": coverage[root]["mapping_events"],
        "mapped_contracts_observed": coverage[root]["mapped_contracts_observed"],
        "mapping_history": histories[root]["mapping_history"],
    }
    for root in subscriptions
}
mapping_and_roll_observations

In [ ]:
# Session counts
from systematic_futures.data.sessions import SessionEngine, reference_session_policies
from systematic_futures.research_lib.coverage_report import session_counts_for_history

session_engine = SessionEngine(reference_session_policies())
session_counts = {
    root: session_counts_for_history(root, histories[root], session_engine)
    for root in subscriptions
}
session_counts

In [ ]:
# DataProbeResult objects
from systematic_futures.research_lib.quantbook_probe import summarize_contract_history

probe_results = tuple(
    summarize_contract_history(root, histories[root])
    for root in subscriptions
)
probe_results

In [ ]:
# Export audited artifacts
from pathlib import Path

from systematic_futures.config.research import (
    PROBE_END_DATE,
    PROBE_START_DATE,
    REFERENCE_MARKETS,
    RESEARCH_RANDOM_SEED,
    lift_1_manifest_configuration,
)
from systematic_futures.domain.enums import ResearchEnvironment
from systematic_futures.ledger.run_manifest import RunManifestBuilder
from systematic_futures.research_lib.export import write_canonical_json
from systematic_futures.research_lib.quantbook_probe import export_probe_results

project_root = Path.cwd()
summary_path = project_root / "artifacts/data_probes/reference_markets_summary.json"
manifest_path = project_root / "artifacts/manifests/lift_1_qc_research_unqualified_manifest.json"
summary_hash = export_probe_results(probe_results, summary_path)
manifest = RunManifestBuilder().build(
    environment=ResearchEnvironment.QC_RESEARCH,
    created_at_utc=datetime.now(UTC),
    configuration=lift_1_manifest_configuration(),
    source_document_paths=(
        project_root / "upload/Institutional_Systematic_Futures_Program_Master_Spec_v1.0(2).docx",
        project_root / "upload/Intraday_Alpha_Capture_Execution_Extension_v1.0_HE(2).docx",
    ),
    dependency_files=(project_root / "pyproject.toml", project_root / "requirements.txt"),
    reference_markets=REFERENCE_MARKETS,
    probe_start_date=PROBE_START_DATE,
    probe_end_date=PROBE_END_DATE,
    lean_version=None,
    repository_revision=None,
    random_seed=RESEARCH_RANDOM_SEED,
)
manifest_artifact_hash = write_canonical_json(manifest, manifest_path)
{
    "summary_path": str(summary_path),
    "summary_hash": summary_hash,
    "manifest_path": str(manifest_path),
    "manifest_hash": manifest.manifest_hash,
    "manifest_artifact_hash": manifest_artifact_hash,
}

## Verified facts, unresolved facts, and limitations

### Verified facts

- Project code fixes the reference registry to ES, ZN, and 6E and preserves continuous and actual-contract identities.
- Exported summaries and manifests use deterministic canonical serialization and lineage hashes.
- The notebook makes no strategy, signal, return, indicator, or P&L calculation.

### Unresolved facts

- Treat actual rows, mappings, expiry/OI coverage, gaps, tick sizes, and multipliers as verified only after this notebook executes successfully in an authorized QC Research environment and the exported artifact is reviewed.
- The notebook's unqualified research manifest keeps the exact LEAN engine version and repository revision as `None` unless direct evidence is supplied; it never overwrites the immutable historical Lift 1 manifest.

### Data limitations and next certification work

- Backwards Ratio values are full-history-adjusted and are not point-in-time signal certified.
- Gap counts are unadjudicated until holidays, maintenance, and early closes are certified.
- Mapping event delivery, actual vendor timing, CFTC timing/revisions, and local/cloud parity require separate evidence before any dataset can become `CERTIFIED_SIGNAL`.